## EE2211 Midterm Exam Tools

In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn as sk
from sklearn.preprocessing import PolynomialFeatures, OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error
try:
    from scipy import stats
except Exception:
    stats = None
try:
    import sympy as sp
except Exception:
    sp = None
np.set_printoptions(suppress=True, precision=6)

In [26]:
def add_bias(X):
    X = np.asarray(X)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    return np.column_stack((np.ones(X.shape[0]), X))

def num_poly_params(d, n): #d is the number of features (excluding bias), n is the degree of the polynomial
    return int(math.comb(d + n, n))

def system_type(X):
    X = np.asarray(X)
    m, n = X.shape
    if m == n:
        return "even-determined"
    if m > n:
        return "over-determined"
    return "under-determined"

def solve_even(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.linalg.solve(X, y)

def solve_over(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return np.linalg.pinv(X.T @ X) @ X.T @ y

def solve_under(X, y):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    return X.T @ np.linalg.pinv(X @ X.T) @ y

def solve_system(X, y):
    st = system_type(X)
    if st == "even-determined":
        if abs(np.linalg.det(X)) > 1e-12:
            w = solve_even(X, y)
        else:
            w = np.linalg.pinv(X) @ y
    elif st == "over-determined":
        w = solve_over(X, y)
    else:
        w = solve_under(X, y)
    return st, w

def mse(y_true, y_pred):
    y_true = np.asarray(y_true).reshape(-1)
    y_pred = np.asarray(y_pred).reshape(-1)
    return np.mean((y_true - y_pred) ** 2)

def rmse(y_true, y_pred):
    return np.sqrt(mse(y_true, y_pred))

def fit_linear_regression(X, y, add_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    Xd = add_bias(X) if add_intercept else X
    w = np.linalg.pinv(Xd.T @ Xd) @ Xd.T @ y
    return w

def predict_linear(X, w, has_intercept=False):
    X = np.asarray(X, dtype=float)
    Xd = X if has_intercept else add_bias(X)
    return Xd @ np.asarray(w, dtype=float)

def matrix_report(A):
    A = np.asarray(A, dtype=float)
    rep = {
        "shape": A.shape,
        "rank": int(np.linalg.matrix_rank(A)),
        "det": float(np.linalg.det(A)) if A.shape[0] == A.shape[1] else None,
        "invertible": bool(A.shape[0] == A.shape[1] and abs(np.linalg.det(A)) > 1e-12),
    }
    if sp is not None:
        M = sp.Matrix(A)
        rep["rref"] = M.rref()[0]
        rep["pivot_cols"] = M.rref()[1]
    return rep

In [8]:
def fit_polynomial_regression(X, y, degree=2, add_intercept=True):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float)
    poly = PolynomialFeatures(degree=degree, include_bias=add_intercept)
    P = poly.fit_transform(X)
    w = np.linalg.pinv(P.T @ P) @ P.T @ y
    return poly, w

def predict_polynomial(X, poly, w):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    P = poly.transform(X)
    return P @ np.asarray(w, dtype=float)

def fit_ridge_primal(X, y, lamb=1e-3, add_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    Xd = add_bias(X) if add_intercept else X
    I = np.eye(Xd.shape[1])
    w = np.linalg.pinv(Xd.T @ Xd + lamb * I) @ Xd.T @ y
    return w

def fit_ridge_dual(X, y, lamb=1e-3, add_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    Xd = add_bias(X) if add_intercept else X
    I = np.eye(Xd.shape[0])
    alpha = np.linalg.pinv(Xd @ Xd.T + lamb * I) @ y
    w = Xd.T @ alpha
    return w

def onehot_regression_fit(X, y, add_intercept=True):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y).reshape(-1, 1)
    enc = OneHotEncoder(sparse_output=False)
    Y = enc.fit_transform(y)
    Xd = add_bias(X) if add_intercept else X
    W = np.linalg.pinv(Xd.T @ Xd) @ Xd.T @ Y
    return enc, W

def onehot_regression_predict(X, W, add_intercept=True):
    X = np.asarray(X, dtype=float)
    Xd = add_bias(X) if add_intercept else X
    scores = Xd @ W
    pred_idx = np.argmax(scores, axis=1)
    return scores, pred_idx

def train_val_mse(X, y, test_size=0.2, random_state=0):
    X = np.asarray(X, dtype=float)
    y = np.asarray(y, dtype=float)
    X_tr, X_va, y_tr, y_va = train_test_split(X, y, test_size=test_size, random_state=random_state)
    w = fit_linear_regression(X_tr, y_tr, add_intercept=True)
    y_hat = predict_linear(X_va, w, has_intercept=False)
    return w, mean_squared_error(y_va, y_hat)

In [9]:
from itertools import product

def normal_prob_between(lo, hi, mu=0.0, sigma=1.0):
    if stats is None:
        raise ImportError("scipy is required for normal distribution calculations")
    return stats.norm.cdf(hi, loc=mu, scale=sigma) - stats.norm.cdf(lo, loc=mu, scale=sigma)

def normal_prob_leq(x, mu=0.0, sigma=1.0):
    if stats is None:
        raise ImportError("scipy is required for normal distribution calculations")
    return stats.norm.cdf(x, loc=mu, scale=sigma)

def enumerate_probability(sample_spaces, event_fn):
    outcomes = list(product(*sample_spaces))
    favorable = [o for o in outcomes if event_fn(o)]
    return len(favorable) / len(outcomes), favorable

def bayes_prob(p_b_given_a, p_a, p_b):
    return (p_b_given_a * p_a) / p_b

def combination(n, r):
    return int(np.math.comb(n, r))

def binomial_pmf(n, r, p):
    return combination(n, r) * (p ** r) * ((1 - p) ** (n - r))

def quick_line_plot(x, y, title="Plot", xlabel="x", ylabel="y"):
    plt.figure(figsize=(6, 4))
    plt.plot(x, y, marker='o')
    plt.title(title)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.grid(True)
    plt.show()

In [10]:
X_demo = np.array([[1, 2], [2, 4], [1, -1]], dtype=float)
y_demo = np.array([0, 0.1, 1], dtype=float)
stype, w_demo = solve_system(X_demo, y_demo)
print("System type:", stype)
print("w:", w_demo)

x1 = np.array([-10, -8, -3, -1, 2, 8], dtype=float)
y1 = np.array([5, 5, 4, 3, 2, 2], dtype=float)
poly3, w3 = fit_polynomial_regression(x1, y1, degree=3)
print("poly degree-3 prediction at x=9:", predict_polynomial(np.array([9.0]), poly3, w3))

p = normal_prob_between(28, 33, mu=30, sigma=1.8) if stats is not None else None
print("P(28 <= X <= 33), X~N(30,1.8^2):", p)

prob_sum7, fav = enumerate_probability([range(1, 7), range(1, 7)], lambda o: sum(o) == 7)
print("Dice probability sum=7:", prob_sum7, "favorable count:", len(fav))

System type: over-determined
w: [ 0.68 -0.32]
poly degree-3 prediction at x=9: [2.466098]
P(28 <= X <= 33), X~N(30,1.8^2): 0.8189493848246799
Dice probability sum=7: 0.16666666666666666 favorable count: 6


In [ ]:
#Q1:
x = np.array([1,2,2.9])
y = np.array([2,4.1,6.1])

x_bias = add_bias(x)
w = solve_over(x_bias, y)
print("Over-determined w:", w)

w_ridge = fit_ridge_primal(x, y, lamb=0.1)
print("Ridge regression w:", w_ridge)

#Value of y when x = 1.5:

print("Predicted ynew: ", w_ridge[0] + w_ridge[1] * 1.5)


Over-determined w: [-0.175092  2.156827]
Ridge regression w: [0.038326 2.047659]
Predicted ynew:  3.1098147362466397


In [24]:
#Q2:
X = np.array([[1,2,3],
              [4,0,6],
              [1,1,0],
              [0,1,2],
              [5,7,-2],
              [-1,4,0]])

X_bias = add_bias(X)

y = np.array([[1,0,0],
              [1,0,0],
              [0,1,0],
              [0,0,1],
              [0,1,0],
              [0,0,1]])

w = solve_over(X_bias, y)
print(w)

print("Predicted class scores:", np.array([1,1,-2,3])@ w)


poly, w_poly = fit_polynomial_regression(X,y,3)
print("Predicted class scores with poly features:", predict_polynomial(np.array([[1,-2,3]]), poly, w_poly))

[[-0.242493  0.927438  0.315055]
 [ 0.015421  0.18138  -0.196801]
 [ 0.090291 -0.193384  0.103093]
 [ 0.216265 -0.275296  0.059032]]
Predicted class scores: [0.24114  0.669697 0.089163]
Predicted class scores with poly features: [[-0.516463  0.900082  2.00518 ]]


In [42]:
#Q3:
print(num_poly_params(3,2))


10


In [30]:
X = np.array([[1,2],[0,1],[2,3]])
print(np.linalg.det(X@X.T))
print(np.linalg.det(X.T@X))

0.0
6.0


In [41]:
#Question 17:
X = np.array([[1,1],[2,1],[1,2],[2,3]])
y = np.array([2,3.1,3.5,4])
X_bias = add_bias(X)
w = solve_over(X_bias,y)
print("Linear regression weights: ", w)

y_pred = X_bias @ w

print("MSE Linear: ", mse(y,y_pred))


poly = PolynomialFeatures(2)
P = poly.fit_transform(X)
w_poly = fit_ridge_primal(P,y,lamb=1.0,add_intercept=False)
print("Polynomial regression weights: ", w)

y_pred_poly = predict_polynomial(X,poly,w_poly)
print("MSE Poly: ", mse(y,y_pred_poly))

Linear regression weights:  [1.29 0.47 0.66]
MSE Linear:  0.11024999999999992
Polynomial regression weights:  [1.29 0.47 0.66]
MSE Poly:  0.18073032552600313


In [53]:
#Question 18:

X = np.array([[2,1,0],[0,3,1],[1,0,3],[3,1,4],[-1,2,1]])
y = np.array([[1],[3],[2],[1],[2]])
poly = PolynomialFeatures(2)
P = poly.fit_transform(X)
enc, w_poly = onehot_regression_fit(P,y,add_intercept=False)
print(w_poly)

x_test = np.array([1,1,2])
x_test = x_test.reshape(1,-1)

x_test_poly = poly.transform(x_test)
y_pred = x_test_poly @ w_poly
print(y_pred)


[[ 0.047728  0.111046 -0.019119]
 [ 0.08119  -0.054782  0.052904]
 [ 0.030855  0.063099 -0.006403]
 [-0.000087  0.106363 -0.009476]
 [ 0.156452  0.099361 -0.103906]
 [ 0.068172 -0.206727  0.106821]
 [-0.044584 -0.219002  0.042244]
 [-0.003111 -0.048571  0.121697]
 [-0.031796 -0.033644 -0.014779]
 [-0.016817  0.131366 -0.003131]]
[[0.161081 0.196323 0.175446]]


In [56]:
def fit_polynomial_regression(X, y, degree=2, add_intercept=True):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    y = np.asarray(y, dtype=float)
    poly = PolynomialFeatures(degree=degree, include_bias=add_intercept)
    P = poly.fit_transform(X)
    w = np.linalg.inv(P.T @ P) @ P.T @ y
    return poly, w

def predict_polynomial(X, poly, w):
    X = np.asarray(X, dtype=float)
    if X.ndim == 1:
        X = X.reshape(-1, 1)
    P = poly.transform(X)
    return P @ np.asarray(w, dtype=float)

In [60]:
#Question 14: 

X = np.array([[12,25,7],[16,30,3],[14,22,10],[10,28,5]])
Y = np.array([[4.2,2.5],[3.8,3.1],[4.5,4.0],[3.0,2.8]])

poly = PolynomialFeatures(2)

P = poly.fit_transform(X)
print(np.linalg.det(P.T@P))

print(P.T@P)

# poly,w = fit_polynomial_regression(X,Y,degree=2,add_intercept=True)
# print(w)


-3.584214292590561e-60
[[      4.      52.     105.      25.     696.    1368.     322.    2793.
      625.     183.]
 [     52.     696.    1368.     322.    9568.   18392.    4236.   36516.
     8020.    2382.]
 [    105.    1368.    2793.     625.   18392.   36516.    8020.   75225.
    15835.    4395.]
 [     25.     322.     625.     183.    4236.    8020.    2382.   15835.
     4395.    1495.]
 [    696.    9568.   18392.    4236.  134688.  254448.   56824.  493664.
   105360.   31460.]
 [   1368.   18392.   36516.    8020.  254448.  493664.  105360.  988092.
   202660.   56820.]
 [    322.    4236.    8020.    2382.   56824.  105360.   31460.  202660.
    56820.   19798.]
 [   2793.   36516.   75225.   15835.  493664.  988092.  202660. 2049537.
   406615.  106725.]
 [    625.    8020.   15835.    4395.  105360.  202660.   56820.  406615.
   106725.   34885.]
 [    183.    2382.    4395.    1495.   31460.   56820.   19798.  106725.
    34885.   13107.]]
